# Patient prior: the two best candidates

MAP estimates of the field of patient 1 for the two best (kappa, tau) candidates of the patient prior studies (`261115_patient_prior_map`, `261115_patient_prior_cv`), shown next to the ground truth: the MAP favourite (0.424, 0.709), with the lowest axial error against the truth, and the cross-validation favourite (0.424, 0.35), with the lowest held-out predictive loss. The estimates are those the studies stored; nothing is optimized here.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyvista as pv

from bayes_cep.mesh.interpolation import NearestNeighborInterpolationStrategy
from bayes_cep.mesh.io import load_pyvista_mesh
from bayes_cep.statistics.axial_statistics import compute_axial_data_diff, wrap_axial_angles
from single_runs.data import PreprocessedData

pv.set_jupyter_backend("trame")

## Settings

`MAP_STUDY_DIR` holds the MAP study (run table and one run directory per cell), `PREPROCESSING_RUNS_DIR` the preprocessing runs it read, one directory per observation seed, named by run id. The first observation seed is `11454bdc`, the second `898e2e01`.

In [2]:
SIMULATIONS_DIR = Path.home() / "5-Simulations"
RAW_DIR = SIMULATIONS_DIR / "261011_patients_raw_data" / "patient_01"
MAP_STUDY_DIR = SIMULATIONS_DIR / "261115_patient_prior_map"
PREPROCESSING_RUNS_DIR = SIMULATIONS_DIR / "261025_prior_selection_preprocessing" / "runs"

PREPROCESSING_RUN_ID = "11454bdc"
CANDIDATES = {"MAP favourite": (0.424, 0.709), "CV favourite": (0.424, 0.35)}  # (kappa, tau)

## Load the data and the stored MAP estimates

In [3]:
mesh = load_pyvista_mesh(RAW_DIR / "mesh.vtu")
data = PreprocessedData.load(PREPROCESSING_RUNS_DIR / PREPROCESSING_RUN_ID / "results")

interpolation_strategy = NearestNeighborInterpolationStrategy()
connectivity = mesh.cells.reshape(-1, 4)[:, 1:]
vertex_to_simplex_matrix = interpolation_strategy.assemble_matrix(mesh.points, connectivity)
ground_truth_simplex = vertex_to_simplex_matrix @ data.ground_truth
prior_mean_simplex = vertex_to_simplex_matrix @ data.prior_mean

run_table = pd.read_parquet(MAP_STUDY_DIR / "summary" / "run_table.parquet")
run_table = run_table[run_table["problem.data_dir"].str.contains(PREPROCESSING_RUN_ID)]
map_estimates = {}
for name, (kappa, tau) in CANDIDATES.items():
    row = run_table[
        np.isclose(run_table["problem.prior.kappa"], kappa)
        & np.isclose(run_table["problem.prior.tau"], tau)
    ].iloc[0]
    map_estimates[name] = np.load(
        MAP_STUDY_DIR / "runs" / row["run_id"] / "results" / "map_estimate.npy"
    )
    print(f"{name} (kappa {kappa}, tau {tau}): run {row['run_id']}, success {row['success']}")

MAP favourite (kappa 0.424, tau 0.709): run d90ba97d, success False
CV favourite (kappa 0.424, tau 0.35): run 522a7b43, success False


## Ground truth and prior mean

Compared on simplices, as the forward model sees the field. The fiber is axial: all angle fields are plotted modulo pi on one cyclic color range.

In [4]:
clim = (-np.pi / 2, np.pi / 2)

for name, field_values in (
    ("ground truth", ground_truth_simplex),
    ("prior mean", prior_mean_simplex),
):
    plot_mesh = mesh.copy()
    plot_mesh.cell_data["angle"] = wrap_axial_angles(field_values)
    plotter = pv.Plotter()
    plotter.add_mesh(plot_mesh, scalars="angle", cmap="hsv", clim=clim)
    plotter.add_text(name)
    plotter.show()

Widget(value='<iframe src="http://localhost:44539/index.html?ui=P_0x7f32de88fe00_0&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:44539/index.html?ui=P_0x7f32844134d0_1&reconnect=auto" class="pyvi…

## MAP estimates vs. ground truth

In [5]:
for name, map_estimate in map_estimates.items():
    map_estimate_simplex = vertex_to_simplex_matrix @ map_estimate
    plot_mesh = mesh.copy()
    plot_mesh.cell_data["angle"] = wrap_axial_angles(map_estimate_simplex)
    plotter = pv.Plotter()
    plotter.add_mesh(plot_mesh, scalars="angle", cmap="hsv", clim=clim)
    plotter.add_text(f"MAP estimate, {name}")
    plotter.show()

Widget(value='<iframe src="http://localhost:44539/index.html?ui=P_0x7f33382efc50_2&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:44539/index.html?ui=P_0x7f3281deac10_3&reconnect=auto" class="pyvi…

## Axial error vs. ground truth

In [ ]:
prior_mean_error = compute_axial_data_diff(prior_mean_simplex, ground_truth_simplex)
print(f"prior mean mean absolute error: {np.degrees(np.mean(np.abs(prior_mean_error))):.2f} deg")

for name, map_estimate in map_estimates.items():
    map_estimate_error = compute_axial_data_diff(
        vertex_to_simplex_matrix @ map_estimate, ground_truth_simplex
    )
    print(f"{name} mean absolute error: {np.degrees(np.mean(np.abs(map_estimate_error))):.2f} deg")
    plot_mesh = mesh.copy()
    plot_mesh.cell_data["error [deg]"] = np.degrees(np.abs(map_estimate_error))
    plotter = pv.Plotter()
    plotter.add_mesh(plot_mesh, scalars="error [deg]", cmap="viridis", clim=(0.0, 90.0))
    plotter.add_text(f"MAP estimate error, {name}")
    plotter.show()